# MASTER THESIS - model V12 summer

This notebook demonstrates the usage of the `DarkGreyBox` models via fitting them with `DarkGreyFit`

## Importation

In [10]:
import sys
import importlib
from pathlib import Path

# go from ...\docs\tutorials up two levels to the repo root (...\darkgreybox)
repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from statsmodels.graphics.tsaplots import plot_pacf

from darkgreybox.models import TiTmCn2R2C_summer_V12
from darkgreybox.fit import darkgreyfit
from docs.tutorials.util.plot import plot
from docs.tutorials.util.error import rmse

Our temporal resolution is 5 minutes

In [11]:
# the duration of a record (in h - 5 minutes)
rec_duration = 5/60

## Data Loading and Preprocessing

The data is loaded from the CSV file 'data_summer' and processed to extract relevant variables for the thermal model.

In [ ]:
input_df = pd.read_csv('./data/data_summer.csv', index_col=0, parse_dates=True)

input_df['Ti_meas'] = input_df['R01_06_TRU01']
input_df['qv'] = input_df['R01_06_FCI01']
input_df['c'] = input_df['R01_06_CO201']            # Use processed CO2 column
input_df['Ta'] = input_df['HTR9_VES01_TUD01']
input_df['Ik'] = input_df['HTR9_VES01_SI01']
input_df['Tsup'] = input_df['HTRK_VEN01_SpTI01_C']


# Set initial conditions
input_df['Ti0'] = input_df['Ti_meas'].iloc[0]
input_df['Tm0'] = input_df['Ti_meas'].iloc[0]
input_df['c0'] = input_df['c'].iloc[0]
input_df['N0'] = 0                                  # We suppose no one in the room at the beginning
input_df['P0_Ti'] = 1.0                             # Initial state covariance for Ti
input_df['P0_Tm'] = 1.0                             # Initial state covariance for Tm
input_df['P0_N'] = 4.0                              # Initial state covariance for N


input_X = input_df[['qv', 'c', 'Ta', 'Ik', 'Ti_meas', 'Tsup', 'Ti0', 'Tm0', 'c0', 'N0', 'P0_Ti', 'P0_Tm', 'P0_N']]
input_y = input_df['Ti_meas']

print(f'Input X shape: {input_X.shape}, input y shape: {input_y.shape}')


Input X shape: (4296, 13), input y shape: (4296,)


## Data Splitting

Initial conditions are set for the state variables at t=0, using the first measurements from the dataset. The data is then split into training (80%) and test (20%) sets for model validation.
`shuffle=False` to preserve the temporal order of the time series data. This is important for thermal modeling where historical context matters.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(input_X, input_y, test_size=0.2, shuffle=False)

print(f'Train: X shape: {X_train.shape}, y shape: {y_train.shape}')
print(f'Test: X shape: {X_test.shape}, y shape: {y_test.shape}')

Train: X shape: (3436, 13), y shape: (3436,)
Test: X shape: (860, 13), y shape: (860,)


## Model Training Parameters and Physical Constants

The parameter ranges are carefully chosen to reflect realistic physical bounds and to ensure the optimizer does not hit artificial limits during fitting. Capacitances and resistances are estimated from building material properties and geometry, while internal gains and constants are set according to typical values found in literature or building standards. This approach balances physical realism with flexibility for model calibration.


In [ ]:
S_room = 11                             #Surface of the room, found on Revit (m²)
A_windows = 5.48                        #Window area found on Revit (m²)
H_room = 2.7                            #Height of the room (m)
V_room = S_room * H_room                #Volume of the room (m³)



# Define training parameters
train_params_TiTmCn2R2C_summer = {
    'Ti0': {'value': input_df.iloc[0]['Ti0'], 'vary': False},               # Initial indoor temperature
    'Tm0': {'value': input_df.iloc[0]['Tm0'], 'vary': False},               # Initial thermal mass temperature
    'c0': {'value': input_df.iloc[0]['c0'], 'vary': False},                 # Initial CO2 concentration
    'N0': {'value': input_df.iloc[0]['N0'], 'vary': False},                 # Initial number of occupants
    
    'Ci': {'value': 5000, 'vary': True, 'min': 1000, 'max': 50000},
    'Cm': {'value': 100000, 'vary': True, 'min': 1000, 'max': 1000000},    # Bigger mass
    'Rim': {'value': 0.005, 'vary': True, 'min': 0.001, 'max': 0.01},      # Mass-air
    'Rout': {'value': 0.15, 'vary': True, 'min': 0.01, 'max': 2.0},         # Envelope
    'q_equip_var': {'value': 80, 'vary': False},                            # Variable equipment gains per person
    'q_equip_const': {'value': 10, 'vary': False},                          # Constant equipment gains
    'V': {'value': S_room*H_room, 'vary': False},                           # Volume of the room
    'S': {'value': S_room, 'vary': False},                                  # Surface of the room
    'A': {'value': A_windows, 'vary': False},                               # Window area
    'c_out': {'value': 400, 'vary': False},                                 # Outdoor CO2 concentration
    'rho_air': {'value': 1.2, 'vary': False},                               # Air density
    'cp_air': {'value': 1000, 'vary': False},                               # Specific heat capacity of air
    'g': {'value': 0.55, 'vary': False},                                    # total solar energy transmittance of the glazing 
    'alpha': {'value': 0.1, 'vary': True, 'min': 0.01, 'max': 0.5},         # CO2 occupancy filter parameter
    'G_base': {'value': 0.016, 'vary': False},                              # Base global radiation factor (for calibration)
    'Met' : {'value': 1.2, 'vary': False},                                  # Metabolic rate (for occupancy gains)
    'alpha_lat' : {'value': 40.0, 'vary': True, 'min': 20.0, 'max': 60.0},  # Latent heat gain factor for occupancy

    'sigma_Ti': {'value': 0.02, 'vary': True, 'min': 1e-4, 'max': 1.0},     # Process noise std for Ti --- IGNORE ---
    'sigma_Tm': {'value': 0.01, 'vary': True, 'min': 1e-4, 'max': 1.0},     # Process noise std for Tm --- IGNORE ---

    'sigma_N': {'value': 0.5, 'vary': True, 'min': 0.01, 'max': 5.0},    
    'sigma_Ti_meas': {'value': 0.3, 'vary': False},                         # Measurement noise std for Ti (from sensor spec)
    'sigma_c': {'value': 50.0, 'vary': False},                              # Measurement noise std for CO2 (from sensor spec)
    
    'P0_Ti': {'value': 1.0, 'vary': False},                                 # Initial state covariance for Ti
    'P0_Tm': {'value': 1.0, 'vary': False},                                 # Initial state covariance for Tm
    'P0_N': {'value': 4.0, 'vary': False},                                  # Initial state covariance for N (assuming we are quite uncertain about initial occupancy)  
}

# Verify stability
print(f"dt = {rec_duration:.4f} hours ({rec_duration * 60:.2f} minutes)")

dt = 0.0833 hours (5.00 minutes)


### 1: Initial conditions
Set up our initial conditions parameters map for testing

### 2: Error metric
To evaluate the performance of our models we will need to define an error metric. Again, there is a wide range of options available from `sklearn`

### 3: Fit method
Under the hood, `DarkGreyBox` uses `lmfit.minimize` to fit the thermal model parameters to the training data by passing it a custom objective function defined by the model class. The fit method used by `lmfit.minimize` can be customised as well. The standard Nelder-Mead and Levenberg-Marquardt methods work well in general. Here we use the Nelder-Mead method.

### 4: Choice of the model

In [ ]:
from lmfit import Parameters, minimize as lmfit_minimize

# ── Convert dict → lmfit Parameters ──────────────────────────────────────────
def build_lmfit_params(params_dict):
    p = Parameters()
    for name, spec in params_dict.items():
        if isinstance(spec, dict):
            p.add(name, **{k: v for k, v in spec.items()
                           if k in ('value', 'vary', 'min', 'max', 'expr')})
        else:
            p.add(name, value=float(spec))
    return p

lmfit_params = build_lmfit_params(train_params_TiTmCn2R2C_summer)

# ── Model instance ────────────────────────────────────────────────────────────
model = TiTmCn2R2C_summer_V12(train_params_TiTmCn2R2C_summer, rec_duration)

# ── Log-likelihood objective (mathematically required for KF model) ───────────
def loglik_objective(params):
    result = model.model(params, X_train)
    z = result.var['z_Ti'][1:]
    S = result.var['S_Ti'][1:]
    return 0.5 * np.sum(z**2 / S + np.log(S))

# ── Fit ───────────────────────────────────────────────────────────────────────
print("Training model...")
fit_result    = lmfit_minimize(loglik_objective, lmfit_params, method=method)
model.result  = fit_result
fitted_params = fit_result.params

# ── Train prediction ──────────────────────────────────────────────────────────
train_results = model.model(fitted_params, X_train)

# ── Test initial conditions + prediction ─────────────────────────────────────
test_params = fitted_params.copy()
test_params['Ti0'].set(value=float(y_test.iloc[0]))
test_params['Tm0'].set(value=float(train_results.var['Tm'][-1]))
test_params['c0'].set(value=float(train_results.var['c'][-1]))
test_params['N0'].set(value=float(train_results.var['N'][-1]))

test_results = model.model(test_params, X_test)
print("Fitting complete!")

# ── Display parameters ────────────────────────────────────────────────────────
if hasattr(model, "result") and hasattr(model.result, "params"):
    display(model.result.params)
else:
    print("Model parameters not available.")


Training model...
Fitting complete!


name,value,initial value,min,max,vary
Ti0,25.3672727,25.36727273,-inf,inf,False
Tm0,25.3672727,25.36727273,-inf,inf,False
c0,433.680000,433.68,-inf,inf,False
N0,0.00000000,0.0,-inf,inf,False
Ci,3975.44235,5000.0,1000.00000,50000.0000,True
Cm,14257.2352,100000.0,1000.00000,1000000.00,True
Rim,0.00896909,0.005,1.0000e-03,0.01000000,True
Rout,0.07867527,0.15,0.01000000,2.00000000,True
q_equip_var,80.0000000,80.0,-inf,inf,False
q_equip_const,10.0000000,10.0,-inf,inf,False


Parameter   | Initial Guess (calc) | Real Initial Guess (used) | Estimated Value
--------------------------------------------------------------------------
Ci         |       1817818.2000 |               5000.0000 |       3975.4423
Cm         |       2032800.0000 |             100000.0000 |      14257.2352
Rim        |             0.0189 |                  0.0050 |          0.0090
Rout       |             0.3636 |                  0.1500 |          0.0787
Train RMSE (Ti): 0.1118 °C
Test  RMSE (Ti): 0.0822 °C
